In [ ]:
# ============================================================
# FOREST FIRE RISK PREDICTION
# EDA + Random Forest / Gradient Boosting (Regression & Classification)
# ============================================================
#
# ------------------------------------------------------------
# PROBLEM STATEMENT -- CASE-BASED SCENARIO
# ------------------------------------------------------------
# Setting: Montesinho Natural Park, Portugal -- Fire Lookout Station
#
# It is a dry Tuesday afternoon in September. Rui, a ranger at Montesinho
# Natural Park, is on duty at the central fire lookout tower. His automatic
# weather station reports the day's readings: temperature of 24.8C, relative
# humidity of 27%, wind at 4.1 m/s, no rain, and Fire Weather Index
# components (FFMC 92.1, DMC 137.0, DC 706.0, ISI 9.8) that describe how dry
# the surface litter, moderate-depth duff, and deep organic layers of the
# forest floor currently are.
#
# Rui's park has 20 lookout stations, and only 6 rangers on shift today. He
# cannot patrol everywhere at once, so every morning he must decide where
# fire risk is highest and deploy his limited patrol teams there.
# Historically, this decision has relied on a ranger's intuition and a fixed
# monthly calendar (e.g. "August is always bad") -- which either wastes
# patrols on quiet days or, worse, leaves a genuinely dangerous area
# unwatched.
#
# THE BUSINESS PROBLEM: Given only the weather-station readings available
# each morning (location coordinates, month, day, the four Fire Weather
# Index components, temperature, relative humidity, wind, and rain -- no
# satellite imagery, no live camera feed), can we build a data-driven
# system that tells Rui's team, station by station, whether today's fire
# risk is Low, Medium, or High -- and, if a fire does start, roughly how
# large an area is likely to burn -- so patrol resources are deployed where
# they matter most?
#
# This mirrors exactly the structure of the UCI Forest Fires dataset
# (Cortez & Morais, 2007): 517 historical fire records from this same park,
# each with the weather readings above and the observed burned area,
# collected specifically to support this kind of early-warning decision.
#
# WHY THIS MATTERS: Forest fires that are caught and responded to early burn
# far smaller areas. A model that reliably flags "High risk" days/locations
# lets a small ranger team act like a much larger one.
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("forestfires.csv")

print("=" * 60)
print("FOREST FIRE RISK PREDICTION")
print("=" * 60)
print("\nDataset Shape:", df.shape)
print("\nFirst 5 Rows:\n", df.head())
print("\nStatistical Summary:\n", df.describe())


# ============================================================
# 2. DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)


# ============================================================
# 3. COLUMNS
# ============================================================

categorical_columns = ['month', 'day']
numerical_columns = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']


# ============================================================
# 4. EXPLORATORY DATA ANALYSIS
# ============================================================

df[numerical_columns + ['area']].hist(figsize=(15, 12), bins=20)
plt.suptitle("Distribution of Numerical Variables", fontsize=16)
plt.tight_layout()
plt.savefig("01_distributions.png")
plt.close()

plt.figure(figsize=(15, 7))
sns.boxplot(data=df[numerical_columns + ['area']])
plt.title("Boxplot of Numerical Variables")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("02_boxplot.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(data=df, x='month', order=df['month'].value_counts().index, ax=axes[0])
axes[0].set_title("Records by Month")
sns.countplot(data=df, x='day', order=df['day'].value_counts().index, ax=axes[1])
axes[1].set_title("Records by Day")
plt.tight_layout()
plt.savefig("03_month_day.png")
plt.close()

df['log_area'] = np.log1p(df['area'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['area'], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Burned Area (raw)")
sns.histplot(df['log_area'], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribution of log(1 + Area)")
plt.tight_layout()
plt.savefig("04_area_distribution.png")
plt.close()

correlation_columns = numerical_columns + ['area']
correlation_matrix = df[correlation_columns].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig("05_correlation.png")
plt.close()

print("\nCorrelation with Burned Area:")
print(correlation_matrix['area'].sort_values(ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.scatterplot(data=df, x='temp', y='area', ax=axes[0, 0]); axes[0, 0].set_title("Temp vs Area")
sns.scatterplot(data=df, x='RH', y='area', ax=axes[0, 1]); axes[0, 1].set_title("RH vs Area")
sns.scatterplot(data=df, x='wind', y='area', ax=axes[1, 0]); axes[1, 0].set_title("Wind vs Area")
sns.scatterplot(data=df, x='rain', y='area', ax=axes[1, 1]); axes[1, 1].set_title("Rain vs Area")
plt.tight_layout()
plt.savefig("06_bivariate.png")
plt.close()

monthly_area = df.groupby('month')['area'].mean().sort_values(ascending=False)
print("\nAverage Burned Area by Month:")
print(monthly_area)

plt.figure(figsize=(10, 5))
monthly_area.plot(kind='bar')
plt.title("Average Burned Area by Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("07_monthly_area.png")
plt.close()

Q1, Q3 = df['area'].quantile(0.25), df['area'].quantile(0.75)
IQR = Q3 - Q1
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df['area'] < lower_bound) | (df['area'] > upper_bound)]
print("\nNumber of Burned Area Outliers:", len(outliers))


# ============================================================
# 5. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(transformers=[
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
    ('numerical', 'passthrough', numerical_columns)
])

X = df.drop(columns=['area', 'log_area'])
y_area = df['area']
y_log_area = df['log_area']


# ============================================================
# 6. REGRESSION — PREDICT BURNED AREA (trained on log(1+area))
# ============================================================

X_train, X_test, ylog_train, ylog_test, yarea_train, yarea_test = train_test_split(
    X, y_log_area, y_area, test_size=0.20, random_state=42
)

random_forest_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42))
])
random_forest_reg.fit(X_train, ylog_train)
rf_pred_area = np.clip(np.expm1(random_forest_reg.predict(X_test)), 0, None)

rf_mae = mean_absolute_error(yarea_test, rf_pred_area)
rf_rmse = np.sqrt(mean_squared_error(yarea_test, rf_pred_area))
rf_r2 = r2_score(yarea_test, rf_pred_area)

print("\n" + "=" * 60)
print("RANDOM FOREST REGRESSOR (log target, evaluated on hectares)")
print("=" * 60)
print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R2   : {rf_r2:.4f}")

gradient_boosting_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42))
])
gradient_boosting_reg.fit(X_train, ylog_train)
gb_pred_area = np.clip(np.expm1(gradient_boosting_reg.predict(X_test)), 0, None)

gb_mae = mean_absolute_error(yarea_test, gb_pred_area)
gb_rmse = np.sqrt(mean_squared_error(yarea_test, gb_pred_area))
gb_r2 = r2_score(yarea_test, gb_pred_area)

print("\n" + "=" * 60)
print("GRADIENT BOOSTING REGRESSOR (log target, evaluated on hectares)")
print("=" * 60)
print(f"MAE  : {gb_mae:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"R2   : {gb_r2:.4f}")

baseline_pred = np.full_like(yarea_test, yarea_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(yarea_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(yarea_test, baseline_pred))

results = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'Baseline (mean)'],
    'MAE': [rf_mae, gb_mae, baseline_mae],
    'RMSE': [rf_rmse, gb_rmse, baseline_rmse],
    'R2': [rf_r2, gb_r2, 0.0]
})
print("\nMODEL COMPARISON (Regression):")
print(results.to_string(index=False))

results.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(9, 5))
plt.title("Regression Error Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("08_regression_comparison.png")
plt.close()


# ============================================================
# 7. CLASSIFICATION — PREDICT FIRE RISK (Low / Medium / High)
# ============================================================

def assign_risk(area):
    if area == 0:
        return "Low"
    elif area <= 10:
        return "Medium"
    else:
        return "High"

df['fire_risk'] = df['area'].apply(assign_risk)
y_risk = df['fire_risk']

print("\nFire Risk Distribution:")
print(y_risk.value_counts())

majority_baseline_acc = y_risk.value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline_acc:.4f}")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_risk, test_size=0.20, random_state=42, stratify=y_risk
)

random_forest_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=300, random_state=42))
])
random_forest_clf.fit(Xc_train, yc_train)
rf_clf_pred = random_forest_clf.predict(Xc_test)

rf_acc = accuracy_score(yc_test, rf_clf_pred)
rf_f1 = f1_score(yc_test, rf_clf_pred, average='macro')

print("\n" + "=" * 60)
print("RANDOM FOREST CLASSIFIER (fire_risk)")
print("=" * 60)
print(f"Accuracy : {rf_acc:.4f}")
print(f"Macro F1 : {rf_f1:.4f}")
print(classification_report(yc_test, rf_clf_pred))

gradient_boosting_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42))
])
gradient_boosting_clf.fit(Xc_train, yc_train)
gb_clf_pred = gradient_boosting_clf.predict(Xc_test)

gb_acc = accuracy_score(yc_test, gb_clf_pred)
gb_f1 = f1_score(yc_test, gb_clf_pred, average='macro')

print("\n" + "=" * 60)
print("GRADIENT BOOSTING CLASSIFIER (fire_risk)")
print("=" * 60)
print(f"Accuracy : {gb_acc:.4f}")
print(f"Macro F1 : {gb_f1:.4f}")
print(classification_report(yc_test, gb_clf_pred))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(yc_test, rf_clf_pred, labels=['Low', 'Medium', 'High'], ax=axes[0], colorbar=False)
axes[0].set_title("Random Forest — Confusion Matrix")
ConfusionMatrixDisplay.from_predictions(yc_test, gb_clf_pred, labels=['Low', 'Medium', 'High'], ax=axes[1], colorbar=False)
axes[1].set_title("Gradient Boosting — Confusion Matrix")
plt.tight_layout()
plt.savefig("09_confusion_matrices.png")
plt.close()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rf = cross_val_score(random_forest_clf, X, y_risk, cv=cv, scoring='accuracy')
cv_gb = cross_val_score(gradient_boosting_clf, X, y_risk, cv=cv, scoring='accuracy')

print(f"\nRandom Forest      5-fold CV accuracy: {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})")
print(f"Gradient Boosting  5-fold CV accuracy: {cv_gb.mean():.4f} (+/- {cv_gb.std():.4f})")


# ============================================================
# 8. FEATURE IMPORTANCE
# ============================================================

feature_names = random_forest_clf.named_steps['preprocessor'].get_feature_names_out()
importances = random_forest_clf.named_steps['model'].feature_importances_

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Features (Random Forest Classifier):")
print(feature_importance.head(10).to_string(index=False))

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature')
plt.title("Top 10 Feature Importances — Random Forest Classifier")
plt.tight_layout()
plt.savefig("10_feature_importance.png")
plt.close()


# ============================================================
# 9. BONUS — BINARY FRAMING (Fire vs No Fire)
# ============================================================

df['fire_occurred'] = (df['area'] > 0).map({True: 'Fire', False: 'No Fire'})
y_binary = df['fire_occurred']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X, y_binary, test_size=0.20, random_state=42, stratify=y_binary
)

log_reg_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])
log_reg_clf.fit(Xb_train, yb_train)
log_reg_pred = log_reg_clf.predict(Xb_test)

binary_acc = accuracy_score(yb_test, log_reg_pred)
binary_majority = y_binary.value_counts(normalize=True).max()

print(f"\nLogistic Regression accuracy (Fire vs No Fire): {binary_acc:.4f}")
print(f"Majority-class baseline for this framing        : {binary_majority:.4f}")


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

best_model_name = "Random Forest Classifier" if rf_acc >= gb_acc else "Gradient Boosting Classifier"
best_acc = max(rf_acc, gb_acc)

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Regression  - Random Forest R2      : {rf_r2:.4f}")
print(f"Regression  - Gradient Boosting R2  : {gb_r2:.4f}")
print(f"Classifier  - Random Forest accuracy: {rf_acc:.4f} (CV: {cv_rf.mean():.4f})")
print(f"Classifier  - Gradient Boosting acc : {gb_acc:.4f} (CV: {cv_gb.mean():.4f})")
print(f"Binary Fire/No-Fire - Logistic Reg  : {binary_acc:.4f}")
print(f"\nBest-performing risk classifier on this split: {best_model_name} ({best_acc:.4f})")
print("\nAll plots saved as PNG files (01_... to 10_...) in the working directory.")

FOREST FIRE RISK PREDICTION

Dataset Shape: (517, 13)

First 5 Rows:
    X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0

Statistical Summary:
                 X           Y        FFMC         DMC          DC         ISI  \
count  517.000000  517.000000  517.000000  517.000000  517.000000  517.000000   
mean     4.669246    4.299807   90.644681  110.872340  547.940039    9.021663   
std      2.313778    1.229900    5.520111   64.046482  248.066192    4.559477   
min      1.000000    2.000000   18.700000    1.100000    7.900000    0.000000   
25%      3.000000    4.000000   90.200000   68.600000  437.700000    6.50000